# EGM722 Assignment - Flood Exposure Analysis

Run `01_data_download.ipynb` first to download the data.

In [ ]:
import numpy as np
import rasterio
import rasterio.merge
from rasterio.vrt import WarpedVRT
from rasterio.warp import calculate_default_transform, reproject, Resampling, transform_bounds
from rasterio.features import shapes
from shapely.geometry import shape
from shapely.validation import make_valid
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path
import glob


In [ ]:
# study area and settings
BBOX = (-96.5, 28.8, -94.5, 30.5) # west, south, east, north
FLOOD_THRESHOLD = 0.05 # backscatter below this = water
CRS = 'EPSG:32614' # UTM Zone 14N for Texas

raw_dir = Path('data/raw')
processed_dir = Path('data/processed')
outputs_dir = Path('outputs')
processed_dir.mkdir(exist_ok=True)
outputs_dir.mkdir(exist_ok=True)

## 1. Load the SAR data

In [ ]:
# find downloaded VV polarisation GeoTIFFs
vv_files = glob.glob(str(raw_dir / '**/*VV*.tif'), recursive=True)
vv_files += glob.glob(str(raw_dir / '*VV*.tif'))
print(f'Found {len(vv_files)} VV file(s)')

In [ ]:
# load all tiles and clip to the study area
datasets = [rasterio.open(f) for f in vv_files]
target_crs = datasets[0].crs

# OPERA tiles have different CRSs, WarpedVRT reprojects each tile so merge() 
# can combine them
vrts = []
for ds in datasets:
    if ds.crs != target_crs:
        vrts.append(WarpedVRT(ds, crs=target_crs))
    else:
        vrts.append(ds)

# rasterio.warp.transform_bounds converts our WGS84 BBOX into the raster CRS
# so merge() knows which portion of the tiles to read
clip_bounds = transform_bounds('EPSG:4326', target_crs, *BBOX)

# merge and clip
mosaic, mosaic_transform = rasterio.merge.merge(vrts, bounds=clip_bounds)
meta = datasets[0].meta.copy()
meta.update({'crs': target_crs, 'height': mosaic.shape[1],
             'width': mosaic.shape[2], 'transform': mosaic_transform})
for ds in datasets:
    ds.close()

clipped_path = processed_dir / 'sar_clipped.tif'
with rasterio.open(clipped_path, 'w', **meta) as dst:
    dst.write(mosaic)
print(f'SAR data loaded and clipped: {clipped_path}')


In [ ]:
# reproject to UTM Zone 14N (EPSG:32614) - the projected CRS for Texas
with rasterio.open(clipped_path) as src:
    transform, width, height = calculate_default_transform(
        src.crs, CRS, src.width, src.height, *src.bounds)
    meta = src.meta.copy()
    meta.update({'crs': CRS, 'transform': transform, 'width': width, 'height': height})
    reprojected_path = processed_dir / 'sar_utm.tif'
    with rasterio.open(reprojected_path, 'w', **meta) as dst:
        reproject(source=rasterio.band(src, 1), destination=rasterio.band(dst, 1),
                  src_crs=src.crs, dst_crs=CRS, resampling=Resampling.bilinear)
print(f'Reprojected: {reprojected_path}')

## 2. Create the flood mask

In [ ]:
# plot backscatter histogram to check the threshold makes sense
with rasterio.open(reprojected_path) as src:
    data = src.read(1).astype(float)
    nodata = src.nodata
    if nodata is not None:
        data[data == nodata] = np.nan

plt.figure(figsize=(8, 4))
plt.hist(data[~np.isnan(data)].ravel(), bins=100, range=(0, 0.5), color='steelblue', edgecolor='none')
plt.axvline(FLOOD_THRESHOLD, color='red', linestyle='--', label=f'Threshold = {FLOOD_THRESHOLD}')
plt.xlabel('Backscatter (linear)')
plt.ylabel('Pixel count')
plt.title('SAR backscatter distribution')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# classify pixels below the threshold as flooded
with rasterio.open(reprojected_path) as src:
    data = src.read(1).astype(float)
    nodata = src.nodata
    meta = src.meta.copy()

if nodata is not None:
    data[data == nodata] = np.nan

flood = (data < FLOOD_THRESHOLD).astype(np.uint8)
flood[np.isnan(data)] = 0

meta.update({'dtype': 'uint8', 'nodata': 255})
flood_mask_path = processed_dir / 'flood_mask.tif'
with rasterio.open(flood_mask_path, 'w', **meta) as dst:
    dst.write(flood, 1)

flood_pixels = int(flood.sum())
with rasterio.open(flood_mask_path) as src:
    pixel_area_m2 = abs(src.transform.a * src.transform.e)
flood_area_km2 = flood_pixels * pixel_area_m2 / 1e6
print(f'Flooded pixels: {flood_pixels}  (~{flood_area_km2:.1f} km²)')

In [ ]:
# convert the flood mask raster into vector polygons
with rasterio.open(flood_mask_path) as src:
    data = src.read(1)
    flood_crs = src.crs
    t = src.transform
    pixel_area_m2 = abs(src.transform.a * src.transform.e)

# rasterio.features.shapes() traces outlines around groups of identical pixels
# and returns (geometry, value) pairs
flood_pixels_arr = (data == 1).astype(np.uint8)
raw_geoms = [shape(geom) for geom, _ in shapes(flood_pixels_arr, mask=flood_pixels_arr, transform=t)]

# remove patches smaller than 4 pixels to speed up process
min_area = pixel_area_m2 * 4
clean_geoms = [g for g in raw_geoms if g.area >= min_area]

# build a GeoDataFrame and simplify polygon edges
# .buffer(0) is used to repair any invalid geometries that can arise 
# after simplification
flood_gdf = gpd.GeoDataFrame(geometry=clean_geoms, crs=flood_crs)
flood_gdf['geometry'] = flood_gdf.geometry.simplify(30).buffer(0)
flood_gdf = flood_gdf[~flood_gdf.geometry.is_empty].reset_index(drop=True)
print(f'Flood extent: {len(flood_gdf)} polygons')
